In [ ]:
# Cell 1 — Load trained model
"""
03_inference.ipynb
==================
Interactive predictions with a trained TGNN-Solv model.

This notebook is the exploratory path. For scripted reports, use
`scripts/evaluate_complete.py`, `scripts/error_analysis.py`,
`scripts/validate_physics.py`, and `scripts/generate_paper_figures.py`.
"""

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
NOTEBOOK_FIG_DIR = FIGURES_DIR / "notebooks"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / "notebooks"
NOTEBOOK_FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
from tgnn_solv.inference import (
    load_model,
    predict_solubility,
    temperature_scan,
    interpret_prediction,
)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
MODEL_PATH = CHECKPOINT_DIR / "tgnn_solv_trained.pt"
model, cfg = load_model(str(MODEL_PATH), DEVICE)
print(f"Loaded {MODEL_PATH} on {DEVICE}")


In [ ]:
# Cell 2 — Single prediction with full report
result = predict_solubility(
    model,
    solute_smiles="CC(=O)Nc1ccc(O)cc1",   # paracetamol
    solvent_smiles="CCO",                   # ethanol
    T=298.15,
)
print(interpret_prediction(result))

In [ ]:
# Cell 3 — Compare multiple solvents
solvents = {
    "Water": "O",
    "Ethanol": "CCO",
    "Methanol": "CO",
    "Acetone": "CC(=O)C",
    "Hexane": "CCCCCC",
    "Toluene": "Cc1ccccc1",
    "DMSO": "CS(=O)C",
    "Chloroform": "ClC(Cl)Cl",
}

solute = "CC(=O)Nc1ccc(O)cc1"  # paracetamol
print(f"{'Solvent':15s} {'x₂':>10s} {'ln(x₂)':>8s} {'γ₂':>8s} {'Ra':>6s}")
print("-" * 50)

for name, smi in solvents.items():
    r = predict_solubility(model, solute, smi, T=298.15)
    print(f"{name:15s} {r['x2']:10.5f} {r['ln_x2']:8.3f} "
          f"{r['gamma_2']:8.2f} {r['Ra']:6.1f}")

In [ ]:
# Cell 4 — Temperature scan with plot
import matplotlib.pyplot as plt

scan = temperature_scan(
    model,
    solute_smiles="CC(=O)Nc1ccc(O)cc1",
    solvent_smiles="CCO",
    T_min=270, T_max=340, n_points=30,
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Solubility vs T
axes[0].plot(scan["T"] - 273.15, scan["x2"], "o-", ms=4)
axes[0].set_xlabel("Temperature (°C)")
axes[0].set_ylabel("x₂ (mole fraction)")
axes[0].set_title("Paracetamol / Ethanol")

# ln(x2) vs 1/T (van 't Hoff style)
axes[1].plot(1000 / scan["T"], scan["ln_x2"], "s-", ms=4, color="green")
axes[1].set_xlabel("1000/T (K⁻¹)")
axes[1].set_ylabel("ln(x₂)")
axes[1].set_title("van 't Hoff plot")

# γ₂ vs T
axes[2].plot(scan["T"] - 273.15, scan["gamma_2"], "^-", ms=4, color="red")
axes[2].set_xlabel("Temperature (°C)")
axes[2].set_ylabel("γ₂")
axes[2].set_title("Activity coefficient")

plt.tight_layout()
plt.show()

# Monotonicity check
#diffs = scan["ln_x2"].diff().dropna()
#print(f"Monotonically increasing: {(diffs >= -0.01).all()}")

In [ ]:
# Cell 5 — Batch prediction
import pandas as pd

systems = [
    ("CC(=O)Nc1ccc(O)cc1", "O", 298.15),       # paracetamol / water
    ("CC(=O)Nc1ccc(O)cc1", "CCO", 298.15),      # paracetamol / ethanol
    ("c1ccc2ccccc2c1", "c1ccccc1", 298.15),      # naphthalene / benzene
    ("c1ccc2ccccc2c1", "O", 298.15),             # naphthalene / water
    ("OC(=O)c1ccccc1", "CCO", 298.15),           # benzoic acid / ethanol
    ("OC(=O)c1ccccc1", "O", 298.15),             # benzoic acid / water
    ("CC(=O)Oc1ccccc1C(=O)O", "CCO", 298.15),   # aspirin / ethanol
    ("CC(=O)Oc1ccccc1C(=O)O", "O", 298.15),     # aspirin / water
]

results = []
for sol, slv, T in systems:
    r = predict_solubility(model, sol, slv, T)
    results.append(r)

df = pd.DataFrame(results)
display_cols = ["solute", "solvent", "x2", "ln_x2", "gamma_2",
                "T_m", "dH_fus", "Ra", "correction"]
print(df[display_cols].to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# Cell 6 — Interpretability: decomposition chart
import matplotlib.pyplot as plt
import numpy as np

# Use results from Cell 5
names = [f"{r['solute'][:20]}\n{r['solvent'][:10]}" for r in results]
phi = [-r["Phi"] for r in results]
lng = [-r["ln_gamma_2"] for r in results]
corr = [r["correction"] for r in results]

x_pos = np.arange(len(results))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x_pos - width, phi, width, label="-Φ (crystal)", color="steelblue")
ax.bar(x_pos, lng, width, label="-ln(γ_2) (interaction)", color="coral")
ax.bar(x_pos + width, corr, width, label="correction", color="gray")

ax.set_xticks(x_pos)
ax.set_xticklabels(names, fontsize=7, rotation=45, ha="right")
ax.set_ylabel("Contribution to ln(x_2)")
ax.set_title("Solubility decomposition: crystal + interaction + correction")
ax.legend()
ax.axhline(0, color="black", lw=0.5)
plt.tight_layout()
plt.show()